# Tool Result Caching [Security - Module 03]

> **MLCourse - Agentic AI - Production Security**

`01_caching_strategies.ipynb` covered caching **LLM calls**. This notebook
covers the layer that module skipped: caching **tool calls**.

An agent that calls a search API, a database, or an internal microservice on
every turn pays that cost every time - even when the same tool is called with
the same arguments seconds apart, which happens constantly in ReAct-style
loops and multi-agent systems. Tool-result caching is a different problem from
LLM-response caching, with its own key shape and its own, much sharper safety
line: **some tools must never be cached at all**, and getting that wrong is
worse than caching a wrong LLM answer, because a stale tool result can drive
an agent to take a real, external action on bad information.

Everything in this notebook runs without an API key.

### What you will learn

1. Why tool calls need their own cache, distinct from the LLM cache.
2. Building a cache key from **tool name + arguments** correctly.
3. The three-way classification: safe to cache, cache briefly, never cache.
4. Per-tool TTL and staleness policy.
5. A caching tool-call wrapper an agent can drop over any tool.

### Setup


In [ ]:
import hashlib
import json
import time
from dataclasses import dataclass, field
from typing import Any, Callable, Dict, Optional

print("Module 03: Tool Result Caching")
print("Deterministic, no API key needed.")


### 1. Why tool calls need their own cache

The LLM cache in `01_caching_strategies.ipynb` keys on the *prompt* - model,
system message, user message, temperature. That is the right key for "will
generation produce the same text." It is the wrong key for a tool call, which
has none of those things: a tool call is a **function name plus arguments**,
called by the agent's runtime rather than by the model, and its result comes
from an external system the LLM never sees until after the call returns.

That difference matters for staleness, too. An LLM's answer to a fixed prompt
at temperature 0 does not change between requests. A tool's answer to fixed
arguments absolutely can: `get_stock_price("AAPL")` returns a different number
every few seconds, `get_current_time()` never returns the same thing twice,
and `send_email(...)` does not have a "same result" at all - it has a *side
effect*, and calling it again is not a cache hit, it is a second email.

So tool caching needs its own key, its own TTL policy per tool, and - the
part LLM caching does not need nearly as badly - a hard **do-not-cache list**.

### 2. The cache key: tool name + arguments, precisely

The key must capture every argument that changes the result, and *only*
those. Get this wrong in either direction and you have a bug:

- **Miss an argument** → two calls that should get different answers share a
  cache entry. A `search(query="refunds", user_id="u1")` result gets served
  to `user_id="u2"`.
- **Include something irrelevant** (a request-id, a timestamp added for
  logging, a mutable object's `repr()`) → every call looks unique and the
  cache never hits at all.

The construction below normalises arguments to a canonical JSON form before
hashing, so argument order and key order never affect the key - `search(a=1,
b=2)` and `search(b=2, a=1)` must hash identically.

### A correct cache key for a tool call


In [ ]:
def tool_cache_key(tool_name: str, **kwargs) -> str:
    """Build a cache key from a tool name and its keyword arguments.

    sort_keys=True makes argument ORDER irrelevant. default=str handles
    argument types json can't serialise natively (falls back to repr, which
    is good enough for a cache key even if not perfectly canonical).
    """
    canonical = json.dumps(kwargs, sort_keys=True, default=str)
    raw = f"{tool_name}:{canonical}"
    return hashlib.sha256(raw.encode()).hexdigest()[:16]


k1 = tool_cache_key("search_docs", query="refund policy", top_k=3)
k2 = tool_cache_key("search_docs", top_k=3, query="refund policy")  # same args, different order
k3 = tool_cache_key("search_docs", query="refund policy", top_k=5)  # different top_k

print(f"k1 (query, top_k=3)          : {k1}")
print(f"k2 (top_k=3, query, reorder) : {k2}   <- same key as k1: {k1 == k2}")
print(f"k3 (top_k=5)                 : {k3}   <- different key: {k1 != k3}")


### The subtler bug: what looks like a fixed argument sometimes isn't

Two argument shapes deserve special attention, because they pass code review
looking harmless and then silently poison a shared cache.

- **A caller identity buried in the arguments.** If `user_id` is part of what
  the tool needs to compute the answer (an account balance, a personalised
  recommendation), it **must** be part of the key. If it is present only for
  logging and does not affect the result, including it is harmless but wastes
  cache hits - decide which case you are in, don't guess.
- **An implicit "now".** A tool that reads `datetime.now()` internally but
  takes no time argument looks pure from the outside. Its cache key never
  changes, so a cached answer from an hour ago gets served as if fresh. This
  is really a special case of the TTL question in section 4 - the fix is
  either to exclude the tool from caching or to give it a short, explicit TTL.

### 3. Which tools are safe to cache?

This is the classification that matters most, and it is a **read/write and
determinism** question, not a "is this tool slow" question.

| Category | Examples | Cache policy |
|---|---|---|
| **Pure / deterministic** | unit conversion, arithmetic, string formatting, a lookup in a static table | cache **indefinitely** (no TTL needed) |
| **Read-only, slow-changing** | a knowledge-base search, a documentation lookup, a product catalog query | cache with a **TTL** matched to how often the source changes |
| **Read-only, time-sensitive** | stock price, weather, `current_time`, live inventory count | cache with a **short** TTL, or don't cache at all |
| **Per-user / stateful read** | "my order history", "my account balance" | cache **per user**, never in a shared cache, short TTL |
| **Write / side-effecting** | send an email, place an order, delete a record, charge a card | **never cache.** Calling it twice is not a cache hit, it is doing the thing twice. |

The single question that sorts a new tool into this table:

> **If I call this tool twice with the same arguments, is getting the first
> result again *exactly* the same as calling it again?**

If yes, it is a caching candidate. If the answer is "no, because the world (or
the system) changed in between," it needs a TTL reflecting how fast that
world changes. If the answer is "no, because calling it *does* something,"
it must never be cached.

### A tool registry that carries its own cache policy


In [ ]:
@dataclass
class ToolPolicy:
    """Declares how a tool's results may be cached, right next to the tool
    itself -- so the policy travels with the tool rather than living in the
    caller's head."""
    cacheable: bool
    ttl_seconds: Optional[float] = None   # None + cacheable=True == forever
    per_user: bool = False
    reason: str = ""


TOOL_POLICIES: Dict[str, ToolPolicy] = {
    "convert_currency":  ToolPolicy(cacheable=True,  ttl_seconds=None,
                                    reason="pure function of its inputs"),
    "search_docs":       ToolPolicy(cacheable=True,  ttl_seconds=3600,
                                    reason="docs change occasionally"),
    "get_stock_price":   ToolPolicy(cacheable=True,  ttl_seconds=5,
                                    reason="moves continuously; a short TTL still saves bursts of repeats"),
    "get_current_time":  ToolPolicy(cacheable=False,
                                    reason="never the same answer twice"),
    "get_my_orders":     ToolPolicy(cacheable=True,  ttl_seconds=30, per_user=True,
                                    reason="per-user data; short TTL, never shared"),
    "send_email":        ToolPolicy(cacheable=False,
                                    reason="side effect -- calling it 'again' sends a second email"),
    "charge_card":       ToolPolicy(cacheable=False,
                                    reason="side effect, and an expensive one to get wrong"),
    "delete_record":     ToolPolicy(cacheable=False,
                                    reason="side effect, irreversible"),
}

print(f"{'tool':<18} {'cacheable':>10} {'ttl':>8} {'per-user':>9}  reason")
print("-" * 90)
for name, p in TOOL_POLICIES.items():
    ttl = f"{p.ttl_seconds}s" if p.ttl_seconds else ("forever" if p.cacheable else "-")
    print(f"{name:<18} {str(p.cacheable):>10} {ttl:>8} {str(p.per_user):>9}  {p.reason}")


### 4. Staleness: TTL is a statement about the world, not the cache

Pick a TTL by asking **how fast the real answer changes**, not by picking a
round number that feels safe. Two examples from the table above, worked
through explicitly:

- `search_docs`: the underlying documentation is edited by humans, rarely, so
  an hour of staleness costs almost nothing and saves a large fraction of
  repeated queries.
- `get_stock_price`: the underlying value changes every few seconds. A 5
  second TTL will not make a trading system correct, but it *will* absorb the
  common case of an agent calling the same price tool three times in one
  reasoning turn - which is a real, frequent pattern, not a hypothetical.

The wrong instinct is "cache everything for 5 minutes, it's simpler." That
policy is unsafe for `get_stock_price` in a way nobody will notice until it
matters, and it throws away hours of validity for `search_docs`.

### 5. Building the caching wrapper

Now the mechanism: a decorator that wraps any tool function, looks up its
policy, and either serves a cached result or calls through and stores the
result - refusing outright for anything marked `cacheable=False`.

### The tool-result cache


In [ ]:
@dataclass
class ToolResultCache:
    """A cache keyed on (tool name, arguments), respecting a per-tool policy.

    Deliberately separate from any LLM-response cache: different key shape,
    different staleness rules, and a hard refusal path the LLM cache doesn't
    need.
    """
    policies: Dict[str, ToolPolicy]
    store: Dict[str, Dict[str, Any]] = field(default_factory=dict)
    stats: Dict[str, int] = field(default_factory=lambda: {"hits": 0, "misses": 0, "refused": 0})

    def _key(self, tool_name: str, user_id: Optional[str], kwargs: dict) -> str:
        policy = self.policies.get(tool_name)
        scoped = dict(kwargs)
        if policy and policy.per_user:
            # The user id becomes PART of the key -- this is what keeps a
            # per-user cache from ever serving one user's data to another.
            scoped["__user__"] = user_id
        return tool_cache_key(tool_name, **scoped)

    def call(self, tool_name: str, fn: Callable, kwargs: dict,
             user_id: Optional[str] = None):
        policy = self.policies.get(tool_name)

        if policy is None:
            # SAFE DEFAULT: an unregistered tool is treated as never-cacheable.
            # The alternative -- caching by default -- silently caches a
            # future send_email()-like tool the day someone forgets to
            # register its policy. Refuse by default; opt in explicitly.
            self.stats["refused"] += 1
            return fn(**kwargs), "MISS (no policy: not cached)"

        if not policy.cacheable:
            self.stats["refused"] += 1
            return fn(**kwargs), f"MISS (never cached: {policy.reason})"

        key = self._key(tool_name, user_id, kwargs)
        entry = self.store.get(key)
        if entry is not None:
            age = time.time() - entry["ts"]
            if policy.ttl_seconds is None or age < policy.ttl_seconds:
                self.stats["hits"] += 1
                return entry["value"], f"HIT (age {age:.2f}s)"
            del self.store[key]        # expired

        self.stats["misses"] += 1
        result = fn(**kwargs)
        self.store[key] = {"value": result, "ts": time.time()}
        return result, "MISS (stored)"


print("ToolResultCache defined")


### Exercising it: a cacheable tool


In [ ]:
def convert_currency(amount: float, to: str) -> str:
    """Pretend currency conversion -- deterministic given its inputs."""
    rates = {"EUR": 0.92, "GBP": 0.79}
    return f"{amount * rates.get(to, 1.0):.2f} {to}"


trc = ToolResultCache(policies=TOOL_POLICIES)

print("Three calls to a pure, cacheable tool:")
for i in range(3):
    result, status = trc.call("convert_currency", convert_currency,
                              {"amount": 100, "to": "EUR"})
    print(f"  call {i + 1}: {result:<12} [{status}]")

print(f"\nstats: {trc.stats}")


### A TTL-bound tool expiring correctly


In [ ]:
_price_call_count = {"n": 0}


def get_stock_price(symbol: str) -> str:
    """Pretend price feed -- a new value every call, so a hit means staleness."""
    _price_call_count["n"] += 1
    return f"{symbol}: ${100 + _price_call_count['n']}"


print("Two rapid calls (should hit the 5s TTL cache):")
for i in range(2):
    result, status = trc.call("get_stock_price", get_stock_price, {"symbol": "ACME"})
    print(f"  call {i + 1}: {result:<14} [{status}]")

print("\nWaiting past the 5s TTL...")
time.sleep(5.2)
result, status = trc.call("get_stock_price", get_stock_price, {"symbol": "ACME"})
print(f"  call 3: {result:<14} [{status}]  <- expired, refetched")


### A never-cached (side-effecting) tool


In [ ]:
_emails_sent = []


def send_email(to: str, body: str) -> str:
    _emails_sent.append((to, body))
    return f"sent to {to} (total sent so far: {len(_emails_sent)})"


print("Calling send_email twice with IDENTICAL arguments:")
for i in range(2):
    result, status = trc.call("send_email", send_email,
                              {"to": "ada@example.com", "body": "Your order shipped."})
    print(f"  call {i + 1}: {result}  [{status}]")

print(f"\nemails actually sent: {len(_emails_sent)}")
print("Both calls went through -- 'never cache' means exactly that. A cache")
print("hit here would have silently swallowed the second email.")


### Per-user isolation: same tool, same args, different users


In [ ]:
_order_db = {"u1": ["order-1", "order-2"], "u2": ["order-9"]}


def get_my_orders(status: str = "all") -> list:
    # In a real tool this would read the CURRENT caller's identity from the
    # request context, not take it as a plain argument -- shown explicitly
    # here via user_id= on trc.call() so the mechanism is visible.
    raise NotImplementedError("see the trc.call(... user_id=...) calls below")


def get_my_orders_impl(user_id: str, status: str = "all") -> list:
    return _order_db.get(user_id, [])


print("Same tool, same 'status' argument, two different users:")
for uid in ("u1", "u2"):
    result, status = trc.call(
        "get_my_orders", lambda status="all", _uid=uid: get_my_orders_impl(_uid, status),
        {"status": "all"}, user_id=uid)
    print(f"  user={uid}: {result}  [{status}]")

print("\nRepeat for u1 -- should HIT its own entry, not u2's:")
result, status = trc.call(
    "get_my_orders", lambda status="all", _uid="u1": get_my_orders_impl(_uid, status),
    {"status": "all"}, user_id="u1")
print(f"  user=u1: {result}  [{status}]")


### Reading the per-user result

`u1` and `u2` called the identically-named tool with the identically-shaped
argument (`status="all"`) and got **different cache entries**, because
`user_id` was folded into the key by `_key()` before hashing. Leaving that out
- treating the arguments alone as the key - is precisely the bug from section
2: it would have served whichever user happened to populate the cache first
to every user after them.

### 6. A malicious or malformed argument is also a caching problem

Tool arguments often come from **the LLM**, which means they can be wrong,
adversarial, or shaped to intentionally collide with another call. Two
concrete risks worth naming.

**Cache-key collision from unnormalised input.** If your key builder does
anything looser than the canonical-JSON approach above - string
concatenation, `str(kwargs)` on a dict whose order Python does not guarantee
across versions - you can get accidental collisions between calls that should
be distinct. This is the tool-caching analogue of the semantic-caching
threshold problem in `01_caching_strategies.ipynb`: a decision boundary that
is a little too loose merges things that should stay separate.

**A cached result reused across a changed authorization context.** If a tool
result was cached while a user had permission to see it, and the user's
permission is later revoked, a naive cache keeps serving the old answer for
the rest of its TTL. This is the same idea as cache poisoning in
`01_caching_strategies.ipynb`, moved to the tool layer: the fix there was
"cache-bust on policy change," and it applies here identically - invalidate
any cached tool result tied to a permission or identity change, don't wait out
the TTL.

### 7. Summary: extending the module's cache-layer table

`01_caching_strategies.ipynb` ended with a table of LLM-response cache types.
Here is the same table, extended with the layer this notebook added.

| Cache | Keys on | Staleness handled by | Security concern |
|---|---|---|---|
| Exact-match (LLM) | model + prompt + params | TTL / none (deterministic at temp 0) | leaking secrets embedded in a prompt |
| Semantic (LLM) | query embedding + threshold | TTL | threshold too loose merges distinct questions |
| **Tool result** | **tool name + arguments (+ user id if per-user)** | **per-tool TTL, chosen from how fast the real value changes** | **caching a side-effecting tool at all; missing an argument that should be in the key** |

Same three rules survive, sharpened for tools:

- The key must capture every input that changes the result - for a tool, that
  includes caller identity whenever the result is per-user.
- TTL is a statement about how fast the tool's underlying reality changes, not
  a fixed constant applied everywhere.
- **Guard first, cache second still applies** - but for tools it starts one
  step earlier: decide whether the tool may be cached **at all** before any
  argument or TTL question is relevant. A write tool never reaches the cache
  layer regardless of what its arguments look like.

### Final summary


In [ ]:
print("=== Tool Result Caching: Summary ===\n")
for line in [
    "Cache key = tool name + canonicalised arguments (+ user id if per-user).",
    "Classify every tool BEFORE writing cache code: pure / slow-changing /",
    "  time-sensitive / per-user / side-effecting.",
    "Side-effecting tools (send, charge, delete, write) are NEVER cached.",
    "TTL reflects how fast the tool's real answer changes, not a fixed default.",
    "Per-user results need the user id IN the key, never a shared entry.",
    "Treat a permission/identity change like a policy change: bust the cache,",
    "  don't wait out the TTL.",
]:
    print("  -", line)
